# Classical vs. neural methods: DoF, error and computation time

In the previous tutorials we approximated the solutions of two problems with
physics-informed neural networks (PINNs): a parametric 2D Laplacian
(stationary, with a parameter $\mu$) and a 2D rotating transport
equation (time-dependent).

Here we put those neural methods side by side with the classical ones, the
finite element method (FEM) and the discontinuous Galerkin method (DG), and
compare them in terms of the number of degrees of freedom (DoF) of the approximation,
the error of the computed solution and the computation time.

The two families of methods are not directly comparable. Classical methods
solve a *linear system* whose size is fixed by the mesh, while neural methods
*optimize* the weights of a network, so their cost is governed by the
optimization process, not by the mesh. The comparison is therefore most meaningful
when we fix a target accuracy and ask how much work each method needs to reach it.

## 1/ A parametric 2D Laplacian on the disk: FEM and PINN

We consider the equation

$$- \mu \Delta u = 4\mu^2\, x\,(2-r^2)\, e^{-r^2}, \qquad r^2 = x^2 + y^2$$

on the unit disk $\Omega := \{(x,y) : x^2 + y^2 \le 1\}$ with parameter
$\mu\in \mathbb{M}=[1,2]$, subject to the Dirichlet condition $u = 0$ on
$\partial\Omega$. The exact solution is

$$ u(x,y,\mu) = \mu \left(e^{-r^2} - e^{-1}\right) x, $$

which vanishes on $\partial\Omega$, is not a polynomial, and is not radial
(the factor $x = r\cos\theta$ breaks the radial symmetry).

We solve this problem *in a monolithic fashion*, without snapshots on $\mu$,
on the whole continuous interval $\mu \in \mathbb{M}=[1,2]$.


### Imports


In [ ]:
# # uncomment the line below if you are on Google Colab
# !pip install scimba

import timeit
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

# --- classical (FEM) ---
from scimba_jax.linear_approximation.basis.analytic_bases import (
    local_lagrange_basis,
    local_lagrange_basis_by_logical,
)
from scimba_jax.linear_approximation.basis.dof_map import UnstructuredNodalDofMap
from scimba_jax.linear_approximation.basis.general_bases import AnalyticBasis
from scimba_jax.linear_approximation.error_analysis import l2_error
from scimba_jax.linear_approximation.galerkin.fem.elliptic_fe_scheme import (
    EllipticFEscheme,
)
from scimba_jax.linear_approximation.meshes.unstructured_mesh import UnstructuredMesh
from scimba_jax.linear_approximation.quad.gauss_quad import UnitSquareTensorized
from scimba_jax.linear_approximation.variables.variables_fe import VariablesFE
from scimba_jax.physical_models.abstract_physical_weak_model import (
    AbstractPhysicalWeakModel,
)
from scimba_jax.physical_models.classical_weakform.laplacian_weak_form import (
    LaplacianWeakForm,
)

# --- neural (PINN) ---
from scimba_jax.domains.meshless_domains.domains_2d import Disk2D
from scimba_jax.nonlinear_approximation.approximation_spaces.approximation_spaces import (
    ApproximationSpace,
)
from scimba_jax.nonlinear_approximation.integration.monte_carlo import (
    DomainSampler,
    TensorizedSampler,
)
from scimba_jax.nonlinear_approximation.integration.monte_carlo_parameters import (
    UniformParametricSampler,
)
from scimba_jax.nonlinear_approximation.networks.mlp import MLP
from scimba_jax.nonlinear_approximation.numerical_solvers.projectors import Projector
from scimba_jax.physical_models.elliptic_pde.laplacians import (
    ParametricLaplacianDirichletND,
)

### The problem data

The right-hand side and the exact solution are functions of $(x,y)$ and $\mu$.


In [ ]:
def f_rhs(xy: jnp.ndarray, mu: jnp.ndarray) -> jnp.ndarray:
    """Right-hand side of the PDE: f = 4 mu^2 x (2 - r^2) exp(-r^2)."""
    x, r2 = xy[..., 0:1], xy[..., 0:1] ** 2 + xy[..., 1:2] ** 2
    return 4 * mu[..., 0:1] ** 2 * x * (2.0 - r2) * jnp.exp(-r2)


def analytic_solution(xy: jnp.ndarray, mu: jnp.ndarray) -> jnp.ndarray:
    """Exact solution: u = mu (exp(-r^2) - exp(-1)) x, zero at r = 1, non-radial."""
    x, r2 = xy[..., 0:1], xy[..., 0:1] ** 2 + xy[..., 1:2] ** 2
    return mu * (jnp.exp(-r2) - jnp.exp(-1.0)) * x

### Method based on a linear approximation space (FEM)

The finite element method is built on three ingredients: a mesh of the
domain, a basis (the shape functions), and the variables that hold the
degrees of freedom. Here the basis is *continuous* across cell interfaces
(Lagrange basis), so the degrees of freedom are shared by neighbouring cells.

We build a factory that, given a mesh resolution, a polynomial order and a
parameter value $\mu$, returns a *solved* FEM assembler for the
Laplacian. The disk meshes
are pre-generated with gmsh (see ``generate_meshes.py``) and stored as
``.npz`` files in the ``meshes/`` folder: ``n_cells`` is the number of
cells per diameter of the disk. Since the mesh and the basis do not depend
on $\mu$ (only the right-hand side does), the factory can also be vmapped
over $\mu$, as we do in 2/ below.


In [ ]:
PHYSICAL_DIM = 2
QUAD_ORDER = 4
OUT_DIM = 1
POLY_ORDER = 1

def find_mesh_dir() -> Path:
    """Locate the folder holding the pre-generated disk meshes.

    Walks up from the current working directory, so the notebooks run both
    from ``tp_scimba/`` and from ``tp_scimba/solutions/``.
    """
    for base in (Path.cwd(), *Path.cwd().parents):
        for cand in (base / "meshes", base / "tp_scimba" / "meshes"):
            if cand.is_dir():
                return cand
    raise FileNotFoundError(
        "could not locate the 'meshes' folder; run generate_meshes.py first"
    )


def load_disk_mesh(n_cells):
    """Load the pre-generated gmsh mesh of the unit disk with n_cells per diameter."""
    data = np.load(find_mesh_dir() / f"disk_{n_cells}.npz")
    return UnstructuredMesh(
        nodes=data["nodes"],
        cells=data["cells"],
        ref_quad=UnitSquareTensorized(dim=PHYSICAL_DIM, order=QUAD_ORDER),
        order=int(data["geom_order"]),
    )


def lagrange(coords, i, mesh):
    return local_lagrange_basis(coords, i, mesh, order=POLY_ORDER, out_dim=OUT_DIM)


def lagrange_by_logical(y, i, mesh):
    return local_lagrange_basis_by_logical(y, i, mesh, order=POLY_ORDER, out_dim=OUT_DIM)


def zero_bc(__x):
    return jnp.zeros(OUT_DIM)


def u_exact(x, mu=1.5):
    """Exact solution at the parameter value mu (default: 1.5, the middle of M)."""
    r2 = x[0] ** 2 + x[1] ** 2
    return (mu * (jnp.exp(-r2) - jnp.exp(-1.0)) * x[0]).reshape(OUT_DIM)


def f_rhs_classical(x, mu=1.5):
    """Right-hand side at the parameter value mu (default: 1.5, the middle of M).

    The classical model solves -Δu = f (LaplacianWeakForm), while the PDE is
    -mu Δu = f_rhs; at fixed mu the right-hand side is therefore f_rhs / mu.
    """
    r2 = x[0] ** 2 + x[1] ** 2
    return 4 * mu * x[0] * (2.0 - r2) * jnp.exp(-r2)


def make_fem(n_cells, poly_order, mu=1.5):
    """Build and solve a FEM assembler for the Laplacian on the disk at parameter mu.

    The mesh and the basis do not depend on mu, only the right-hand side does,
    so this factory can be vmapped over mu (see the section below).
    """
    global POLY_ORDER
    POLY_ORDER = poly_order
    mesh = load_disk_mesh(n_cells)
    basis = AnalyticBasis(
        nb_basis=(poly_order + 1) ** PHYSICAL_DIM,
        out_dim=OUT_DIM,
        mesh=mesh,
        local_basis=lagrange,
        local_basis_by_logical=lagrange_by_logical,
        basis_type="scalar",
    )
    variables = VariablesFE(
        basis=basis, nb_variables=OUT_DIM, dof_map=UnstructuredNodalDofMap
    )
    pde = LaplacianWeakForm(dim=PHYSICAL_DIM, f=lambda x: f_rhs_classical(x, mu))
    model = AbstractPhysicalWeakModel.from_weak_form(pde, dirichlet=zero_bc)
    assembler = EllipticFEscheme(model, variables)
    return EllipticFEscheme.solve(assembler, max_iter=1)

### Method based on a nonlinear approximation spaces (PINN)

The PINN is built exactly as in the previous tutorials: a domain, a sampler,
a physical model, an approximation space (a MLP), and a
projector that trains the network. Here the Dirichlet condition is enforced
*strongly*: the network output is multiplied by $1 - r^2$ through a
post-processing, so the approximation vanishes on the circle $r = 1$ by
construction.


In [ ]:
domain_x = Disk2D([0.0, 0.0], 1.0, is_main_domain=True)
domain_mu = [(1.0, 2.0)]
sampler = TensorizedSampler(
    [DomainSampler(domain_x), UniformParametricSampler(domain_mu)], bc=False
)

model_pinn = ParametricLaplacianDirichletND(domain_x, f_rhs, bc="strong")


def post_processing(approx: jnp.ndarray, xy: jnp.ndarray, mu: jnp.ndarray) -> jnp.ndarray:
    r"""Post-processing enforcing the Dirichlet condition strongly.

    We write v_theta = A \circ u_theta with A(u; x, y) = (1 - r^2) u, which vanishes
    on the boundary of the unit disk, and u_theta is the network output.
    """
    r2 = xy[0] ** 2 + xy[1] ** 2
    return approx * (1.0 - r2)


def make_pinn(hidden_sizes, n_epochs, n_colloc, n_bc_colloc=0, seed=0,
              matrix_regularization=1e-8, adaptive_matrix_regularization=True):
    """Build and train a PINN for the parametric Laplacian.

    The Gram matrix of the ENG optimizer is regularized with
    ``matrix_regularization``, and the step size is chosen by a strong-Wolfe
    line search.
    """
    key = jax.random.PRNGKey(seed)
    nn = MLP(
        in_size=3,  # x, y, mu
        out_size=1,
        hidden_sizes=hidden_sizes,
        key=key,
    )
    space = ApproximationSpace(
        {"x": 2},
        [(nn, "scalar", None)],
        model_type="x_mu",
        post_processing=post_processing,  # strong Dirichlet BC: multiply by 1 - r^2
    )
    pinn = Projector(
        model_pinn,
        space,
        sampler,
        adaptive_matrix_regularization=adaptive_matrix_regularization,
        matrix_regularization=matrix_regularization,
        linesearch="strong-wolfe",
    )
    start = timeit.default_timer()
    key, pinn = pinn.project(key, space, n_epochs, n_colloc, n_bc_colloc)
    elapsed = timeit.default_timer() - start
    return pinn, elapsed

### Comparing DoF, error and time

We test FEM over a range of mesh resolutions and polynomial orders, and the
PINN over a range of network sizes and training budgets. For each
configuration we record the number of degrees of freedom, the relative $L^2$
error and the computation time.


In [ ]:
# --- classical sweep ---
fem_results = []
N_CELLS_SWEEP = [4, 8, 16, 32]
POLY_ORDER_SWEEP = [1, 2]

# --- pinn sweep ---
pinn_results = []
PINN_HIDDEN_SWEEP = [[12] * 2, [16] * 2, [16] * 3]
PINN_EPOCHS_SWEEP = [50, 400]
PINN_N_COLLOC = 1000

# --- perform the sweeps ---

for n_cells in N_CELLS_SWEEP:
    for poly_order in POLY_ORDER_SWEEP:
        t0 = timeit.default_timer()
        asm = make_fem(n_cells, poly_order)
        t_fem = timeit.default_timer() - t0
        ndof = asm.variables.dofsl.size
        err = float(l2_error(asm, u_exact, relative=True))
        fem_results.append((poly_order, ndof, err, t_fem))
        print(f"FEM  {n_cells}x{n_cells} Q{poly_order}: ndof={ndof:>6}  "
              f"err={err:.3e}  time={t_fem:.3f}s")

for hidden in PINN_HIDDEN_SWEEP:
    for n_epochs in PINN_EPOCHS_SWEEP:
        pinn, elapsed = make_pinn(hidden, n_epochs, n_colloc=PINN_N_COLLOC)
        ndof = pinn.space.ndof
        key = jax.random.PRNGKey(42)
        n_pts = 20000
        r = jnp.sqrt(jax.random.uniform(key, (n_pts, 1)))
        theta = jax.random.uniform(key, (n_pts, 1), minval=0.0, maxval=2.0 * jnp.pi)
        xy = jnp.concatenate([r * jnp.cos(theta), r * jnp.sin(theta)], axis=1)
        mu = jax.random.uniform(key, (n_pts, 1), minval=1.0, maxval=2.0)
        u_num = pinn.evaluate(xy, mu)
        u_ex = analytic_solution(xy, mu)
        err = float(jnp.linalg.norm(u_num - u_ex) / jnp.linalg.norm(u_ex))
        pinn_results.append((n_epochs, ndof, err, elapsed))

### Plotting the results

We plot the error against the number of degrees of freedom, and the
computation time against the error, for each method.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- error vs DoF ---
ax = axes[0]
for poly_order in POLY_ORDER_SWEEP:
    rs = [r for r in fem_results if r[0] == poly_order]
    ax.loglog([r[1] for r in rs], [r[2] for r in rs],
              "o-", label=f"FEM Q{poly_order}")
for n_epochs in PINN_EPOCHS_SWEEP:
    rs = [r for r in pinn_results if r[0] == n_epochs]
    ax.loglog([r[1] for r in rs], [r[2] for r in rs],
              "^-", label=f"PINN {n_epochs} epochs")
ax.set_xlabel("Number of degrees of freedom")
ax.set_ylabel("Relative L2 error")
ax.set_title("Error vs. DoF")
ax.legend()
ax.grid(True, which="both", alpha=0.3)

# --- time vs error ---
ax = axes[1]
for poly_order in POLY_ORDER_SWEEP:
    rs = [r for r in fem_results if r[0] == poly_order]
    ax.loglog([r[2] for r in rs], [r[3] for r in rs],
              "o-", label=f"FEM Q{poly_order}")
for n_epochs in PINN_EPOCHS_SWEEP:
    rs = [r for r in pinn_results if r[0] == n_epochs]
    ax.loglog([r[2] for r in rs], [r[3] for r in rs],
              "^-", label=f"PINN {n_epochs} epochs")
ax.set_xlabel("Relative L2 error")
ax.set_ylabel("Computation time (s)")
ax.set_title("Time vs. error")
ax.legend()
ax.grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.show()

## 2/ Many parameter values: looped vs. vectorized classical solves, and PINN evaluation

In 1/ we treated the parameter $\mu$ in a monolithic way: the classical methods
were run at a single value ($\mu = 1.5$) and the PINN was trained on the whole
interval $\mathbb{M} = [1,2]$ at once. In practice one often rather wants the
solution at a *finite set* of parameter values $\mu_1, \dots, \mu_N$ (a
multi-query setting: $N$ snapshots, $N$ design points, ...).

For the classical methods there are then two strategies. Either we *loop* over
the parameter values and call the factory of 1/ once per $\mu$, or we
*vectorize*: the mesh and the basis do not depend on $\mu$, only the right-hand
side does, so `jax.vmap` can batch the whole factory over $\mu$; the assembly
and the linear solve are then compiled once and applied to the $N$ right-hand
sides at once. The PINN needs no per-$\mu$ work at all: it is trained once on
the whole interval, then simply *evaluated* at the $N$ values.

Below we compare, for $N \in \{1, 10, 100, 1000, 5000\}$, the total time needed
to get the solution at $N$ parameter values with each strategy (the looped
classical solves beyond $N = 10$ are extrapolated from $N = 1$ and $N = 10$,
their cost being one independent solve per $\mu$), and we compute the number
$N^*$ of parameter values from which the vectorized solve recoups its overhead
compared to the loop.


### Setup


In [ ]:
N_MU_SWEEP = [1, 10, 100, 1000, 5000]
N_CELLS_PARAM = 16
POLY_PARAM = 2
N_ERR = 1000
# number of parameter values at which the looped solves are measured (the
# remaining ones are extrapolated, the loop cost being linear in N)
N_MU_LOOP_MEASURED = [1, 10]

# --- one PINN, trained once on the whole interval M = [1, 2] ---
PINN_PARAM_HIDDEN = [16] * 3
PINN_PARAM_EPOCHS = 100
pinn_param, t_pinn_train = make_pinn(
    PINN_PARAM_HIDDEN, PINN_PARAM_EPOCHS, n_colloc=PINN_N_COLLOC
)
print(f"PINN trained once on M = [1, 2]: ndof={pinn_param.space.ndof}, "
      f"training time={t_pinn_train:.3f}s")

# fixed cloud of points at which the PINN is evaluated for each mu
key = jax.random.PRNGKey(42)
N_PTS_EVAL = 2000
r_eval = jnp.sqrt(jax.random.uniform(key, (N_PTS_EVAL, 1)))
theta_eval = jax.random.uniform(key, (N_PTS_EVAL, 1), minval=0.0, maxval=2.0 * jnp.pi)
XY_EVAL = jnp.concatenate(
    [r_eval * jnp.cos(theta_eval), r_eval * jnp.sin(theta_eval)], axis=1
)

### Warm-up: compile each model once before measuring

The first call to a jitted function (assembly, linear solve, vmapped batch,
network evaluation) pays the XLA compilation cost. Without a warm-up this cost
is charged to the first measurement and makes the $N = 1$ runs inconsistent
with the larger $N$ (the vectorized solve at $N = 1$ can appear more expensive
than at $N = 10$). We therefore evaluate each model once, for every batch size
used below, before measuring anything.


In [ ]:
# one looped solve (compiles the assembly + linear solve)
make_fem(N_CELLS_PARAM, POLY_PARAM, 1.5)

for N in [*N_MU_SWEEP, N_ERR]:
    mus_warm = jnp.linspace(1.0, 2.0, N)
    jax.vmap(lambda mu: make_fem(N_CELLS_PARAM, POLY_PARAM, mu))(mus_warm)
    # one batched PINN evaluation at this batch size (compiles the evaluator)
    mu_b_warm = jnp.broadcast_to(mus_warm[:, None, None], (N, N_PTS_EVAL, 1))
    mu_b_warm = mu_b_warm.reshape(N * N_PTS_EVAL, 1)
    xy_b_warm = jnp.broadcast_to(XY_EVAL[None], (N, N_PTS_EVAL, 2))
    xy_b_warm = xy_b_warm.reshape(N * N_PTS_EVAL, 2)
    pinn_param.evaluate(xy_b_warm, mu_b_warm)

print("warm-up done: every model has been compiled once")

### Running the comparison

For each $N$ we measure the total time of each strategy and the relative $L^2$
error of the computed solutions against the exact solution at each $\mu$. For
the vectorized solves we also check that they coincide with the looped ones.


In [ ]:
# results_param[method][N] = total time (s) to get the solution at the N values of mu
# errors_param[method][N]  = mean relative L2 error over the N values of mu
METHODS_PARAM = ("FEM loop", "FEM vmap", "PINN eval")
results_param = {m: {} for m in METHODS_PARAM}
errors_param = {m: {} for m in METHODS_PARAM}

# reference assembler on the same mesh and basis, used to read the error of
# the vmapped dofsl one mu at a time
asm_fem_ref = make_fem(N_CELLS_PARAM, POLY_PARAM, 1.5)


def classical_errors(asm_ref, asm_batch, mus):
    """Mean relative L2 error of a batched (vmapped) assembler over the values of mu."""
    errs = []
    for i, mu in enumerate(mus):
        asm_ref.variables.dofsl = asm_batch.variables.dofsl[i]
        errs.append(
            float(l2_error(asm_ref, lambda x, m=float(mu): u_exact(x, m), relative=True))
        )
    return float(np.mean(errs))


for N in N_MU_SWEEP:
    mus = jnp.linspace(1.0, 2.0, N)

    # --- looped classical solves: one call to the factory per mu ---
    if N in N_MU_LOOP_MEASURED:
        t0 = timeit.default_timer()
        asms_fem = [make_fem(N_CELLS_PARAM, POLY_PARAM, mu) for mu in mus]
        t_fem_loop = timeit.default_timer() - t0
        results_param["FEM loop"][N] = t_fem_loop
        errors_param["FEM loop"][N] = float(np.mean([
            float(l2_error(a, lambda x, m=float(mu): u_exact(x, m), relative=True))
            for a, mu in zip(asms_fem, mus)
        ]))

    # --- vectorized classical solves: vmap the factory over mu ---
    t0 = timeit.default_timer()
    asm_fem_vec = jax.vmap(lambda mu: make_fem(N_CELLS_PARAM, POLY_PARAM, mu))(mus)
    t_fem_vmap = timeit.default_timer() - t0
    results_param["FEM vmap"][N] = t_fem_vmap
    if N in N_MU_LOOP_MEASURED:
        errors_param["FEM vmap"][N] = classical_errors(asm_fem_ref, asm_fem_vec, mus)
        # sanity check: the vmapped solves coincide with the looped ones
        d_fem = float(jnp.max(jnp.abs(
            asm_fem_vec.variables.dofsl
            - jnp.stack([a.variables.dofsl for a in asms_fem])
        )))
        print(f"N={N:3d}  max|dofsl vmap - dofsl loop|: FEM={d_fem:.2e}")
    else:
        # the error does not depend on mu here (the problem is linear in mu),
        # so we carry over the one measured at the largest measured N
        errors_param["FEM vmap"][N] = errors_param["FEM vmap"][N_MU_LOOP_MEASURED[-1]]

    # --- PINN: evaluate the trained network at the N values of mu ---
    mu_b = jnp.broadcast_to(mus[:, None, None], (N, N_PTS_EVAL, 1)).reshape(
        N * N_PTS_EVAL, 1
    )
    xy_b = jnp.broadcast_to(XY_EVAL[None], (N, N_PTS_EVAL, 2)).reshape(
        N * N_PTS_EVAL, 2
    )
    t0 = timeit.default_timer()
    u_num = pinn_param.evaluate(xy_b, mu_b).reshape(N, N_PTS_EVAL, 1)
    t_pinn_eval = timeit.default_timer() - t0
    results_param["PINN eval"][N] = t_pinn_eval
    u_ex = analytic_solution(xy_b, mu_b).reshape(N, N_PTS_EVAL, 1)
    errors_param["PINN eval"][N] = float(
        jnp.linalg.norm(u_num - u_ex) / jnp.linalg.norm(u_ex)
    )

    msg = f"N={N:3d}:"
    if N in N_MU_LOOP_MEASURED:
        msg += f"  FEM loop={t_fem_loop:.3f}s"
    msg += f"  FEM vmap={t_fem_vmap:.3f}s  PINN eval={t_pinn_eval:.3f}s"
    print(msg)

### Error statistics at 1000 parameter values (vectorized)

Finally we look at the accuracy of each method when the solution is wanted at
$N = 1000$ parameter values. The classical solves are vectorized over $\mu$
(looping over 1000 values would take minutes); the PINN is simply evaluated at
the 1000 values. For each method we report the mean, min and max of the
relative $L^2$ error over the 1000 values of $\mu$.


In [ ]:
import copy

mus_err = jnp.linspace(1.0, 2.0, N_ERR)

# --- vectorized classical solves at N = N_ERR (timed; warmed up above) ---
t0 = timeit.default_timer()
asm_fem_vec_1k = jax.vmap(lambda mu: make_fem(N_CELLS_PARAM, POLY_PARAM, mu))(mus_err)
t_fem_1k = timeit.default_timer() - t0

# --- per-mu relative L2 errors of a vmapped batch, computed all at once ---
# quadrature data of the shared mesh
w_all, x_all = asm_fem_ref.variables.mesh.evaluate_mesh_weights_points()


def batched_l2_errors(variables_ref, dofsl_batch, mus):
    """Relative L2 error at the quadrature points, for each mu of a vmapped batch."""
    def one(dofs, mu):
        v = copy.copy(variables_ref)  # shallow copy: only the dofsl leaf changes
        v.dofsl = dofs
        u_h = v.evaluate_quad(x_all)
        u_ex = jax.vmap(jax.vmap(lambda x: u_exact(x, mu)))(x_all).reshape(u_h.shape)
        num = jnp.sqrt(jnp.sum(w_all[:, :, None] * (u_h - u_ex) ** 2))
        den = jnp.sqrt(jnp.sum(w_all[:, :, None] * u_ex ** 2))
        return num / den

    return jax.vmap(one)(dofsl_batch, mus)


errs_fem_1k = batched_l2_errors(
    asm_fem_ref.variables, asm_fem_vec_1k.variables.dofsl, mus_err
)

# --- PINN evaluated at the N_ERR values of mu ---
mu_b = jnp.broadcast_to(mus_err[:, None, None], (N_ERR, N_PTS_EVAL, 1))
mu_b = mu_b.reshape(N_ERR * N_PTS_EVAL, 1)
xy_b = jnp.broadcast_to(XY_EVAL[None], (N_ERR, N_PTS_EVAL, 2))
xy_b = xy_b.reshape(N_ERR * N_PTS_EVAL, 2)
t0 = timeit.default_timer()
u_num = pinn_param.evaluate(xy_b, mu_b).reshape(N_ERR, N_PTS_EVAL, 1)
t_pinn_1k = timeit.default_timer() - t0
u_ex = analytic_solution(xy_b, mu_b).reshape(N_ERR, N_PTS_EVAL, 1)
errs_pinn_1k = (
    jnp.linalg.norm(u_num - u_ex, axis=(1, 2)) / jnp.linalg.norm(u_ex, axis=(1, 2))
)

# --- summary: solve/eval times, and error statistics as a table (a str) ---
print(f"N = {N_ERR} parameter values (classical solves vectorized):")
print(f"  FEM vmap: solve time {t_fem_1k:.3f}s")
print(f"  PINN eval: {t_pinn_1k:.3f}s (on top of the single training)")

# one row per method: mean / min / max of the relative L2 error over the N_ERR values of mu
error_stats_table = (
    f"relative L2 error over N = {N_ERR} values of mu (classical solves vectorized)\n"
    f"{'method':<12}{'mean':>12}{'min':>12}{'max':>12}\n"
    + "\n".join(
        f"{name:<12}{float(jnp.mean(errs)):12.3e}"
        f"{float(jnp.min(errs)):12.3e}{float(jnp.max(errs)):12.3e}"
        for name, errs in (("FEM vmap", errs_fem_1k), ("PINN eval", errs_pinn_1k))
    )
)
print(error_stats_table)

# store the mean errors at N = N_ERR in the comparison results (summary below)
errors_param["FEM vmap"][N_ERR] = float(jnp.mean(errs_fem_1k))
errors_param["PINN eval"][N_ERR] = float(jnp.mean(errs_pinn_1k))

### How many parameter values to recoup the cost of the vectorization?

Each call to the factory rebuilds the jitted assembly and linear solve, so the
looped strategy pays the compilation cost $N$ times, while the vmapped strategy
pays it once and batches the $N$ solves. Fitting $t(N) = a + b\,N$ to the
measured times of each strategy, the vectorized solve becomes cheaper than the
loop from
$$N^* = \frac{a_{\mathrm{vmap}} - a_{\mathrm{loop}}}{b_{\mathrm{loop}} - b_{\mathrm{vmap}}}$$
parameter values on. The PINN, for its part, has a fixed cost (one training on
the whole interval) plus a nearly free evaluation at each $\mu$.


In [ ]:
def fit_line(ns, ts):
    """Least-squares fit of t(N) = a + b*N; returns (a, b)."""
    b, a = np.polyfit(ns, ts, 1)
    return float(a), float(b)


# extrapolate the looped times over the whole sweep (one solve per mu: linear in N)
for method in ("FEM loop",):
    a, b = fit_line(
        N_MU_LOOP_MEASURED, [results_param[method][n] for n in N_MU_LOOP_MEASURED]
    )
    for N in N_MU_SWEEP:
        if N not in results_param[method]:
            results_param[method][N] = a + b * N
            errors_param[method][N] = errors_param[method][N_MU_LOOP_MEASURED[-1]]
    print(f"{method}: t(N) ~ {a:.3f} + {b:.4f} * N (measured at N={N_MU_LOOP_MEASURED})")

print()
n_stars = {}
for classical in ("FEM",):
    a_loop, b_loop = fit_line(
        N_MU_LOOP_MEASURED,
        [results_param[f"{classical} loop"][n] for n in N_MU_LOOP_MEASURED],
    )
    a_vec, b_vec = fit_line(
        N_MU_SWEEP, [results_param[f"{classical} vmap"][N] for N in N_MU_SWEEP]
    )
    n_star = (a_vec - a_loop) / (b_loop - b_vec)
    n_stars[classical] = n_star
    print(f"{classical}: loop t(N) ~ {a_loop:.3f} + {b_loop:.4f} * N, "
          f"vmap t(N) ~ {a_vec:.3f} + {b_vec:.4f} * N")
    print(f"  -> vectorized {classical} recoups its cost from "
          f"N* = {n_star:.1f} parameter values on")

print()
print(f"PINN: one training ({t_pinn_train:.3f}s), then evaluation at N values of mu:")
for N in N_MU_SWEEP:
    t_eval = results_param["PINN eval"][N]
    print(f"  N={N:3d}: eval={t_eval:.3f}s, total={t_pinn_train + t_eval:.3f}s, "
          f"err={errors_param['PINN eval'][N]:.3e}")

print()
print("summary: total time (s) / mean relative L2 error")
for N in N_MU_SWEEP:
    cells = "  ".join(
        f"{m}={results_param[m][N]:.3f}s,{errors_param[m][N]:.2e}"
        for m in METHODS_PARAM
    )
    print(f"N={N:3d}: {cells}")

### Plotting the results

We plot the total time against the number of parameter values $N$ (log-log),
with the fitted lines extrapolated up to $N = 1000$, and the break-even points
$N^*$.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
N_GRID = np.array([1, 3, 10, 30, 100, 300, 1000, 3000, 5000, 10000], dtype=float)

styles = {
    "FEM loop": ("o-", "tab:blue"),
    "FEM vmap": ("o--", "tab:blue"),
}
for method, (ls, color) in styles.items():
    ns = sorted(results_param[method])
    ts = [results_param[method][n] for n in ns]
    ax.loglog(ns, ts, ls, color=color, label=method)
    a, b = fit_line(ns, ts)
    ax.loglog(N_GRID, a + b * N_GRID, ":", color=color, alpha=0.6)

# PINN: fixed training cost + evaluation time at the N values of mu
ns = sorted(results_param["PINN eval"])
ts = [t_pinn_train + results_param["PINN eval"][n] for n in ns]
ax.loglog(ns, ts, "^-", color="tab:green", label="PINN (train once + eval)")
a, b = fit_line(ns, ts)
ax.loglog(N_GRID, a + b * N_GRID, ":", color="tab:green", alpha=0.6)

for classical, n_star in n_stars.items():
    ax.axvline(n_star, color="gray", ls="-.", alpha=0.7)
    ax.text(n_star, 0.03, f"{classical} N*={n_star:.0f}",
            transform=ax.get_xaxis_transform(), rotation=90, va="bottom",
            fontsize=8, color="gray")

ax.set_xlabel("Number of parameter values $N$")
ax.set_ylabel("Total computation time (s)")
ax.set_title("Looped vs. vectorized classical solves, and PINN evaluation")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

## 3/ A 2D rotating transport equation: DG, PINN, Discrete PINN and Neural SL

We now consider the time-dependent advection equation

$$\partial_t u + \mathbf{a} \cdot \nabla u = 0 \quad \text{in } \Omega \times (0, T)$$

with the rigid rotation velocity field $\mathbf{a}(x) = (-2\pi x_2, 2\pi x_1)$
on the unit disk $\Omega = \{(x,y) : x^2 + y^2 \le 1\}$, and initial condition

$$u_0(x,y) = \exp\!\big(-150\,((x - 0.3)^2 + y^2)\big).$$

The exact solution is the initial bump rotated rigidly by the angle $2\pi t$:

$$u(x,y,t) = u_0\big(R(-2\pi t)\, (x,y)\big),$$

where $R(\theta)$ is the rotation matrix.

We compare four methods: a classical DG method (time-stepped with a
Runge-Kutta scheme), a PINN (the standard physics-informed neural network),
a Discrete PINN (Runge-Kutta in time, a PINN at each stage) and a Neural
Semi-Lagrangian method (the network is fitted to the solution at the foot of
the characteristic at each time step).


### Imports for the transport problem


In [ ]:
from scimba_jax.domains.meshless_domains.domains_2d import Disk2D
from scimba_jax.nonlinear_approximation.integration.monte_carlo_time import (
    UniformTimeSampler,
)
from scimba_jax.nonlinear_approximation.numerical_solvers.discrete_pinns import (
    DiscretePINN,
)
from scimba_jax.nonlinear_approximation.numerical_solvers.neural_semi_lagrangian import (
    NeuralSemiLagrangian,
)
from scimba_jax.physical_models.temporal_pde.advection import (
    AdvectionND,
    SteadyAdvectionND,
)
from scimba_jax.time_discrete.butcher_tableau import build_alexander_tableau

# --- classical (DG) ---
from scimba_jax.linear_approximation.basis.analytic_bases import (
    local_taylor_basis,
    local_taylor_basis_by_logical,
)
from scimba_jax.linear_approximation.galerkin.dg.flux import UpwindFlux
from scimba_jax.linear_approximation.galerkin.dg.time_discrete_dg_scheme import (
    TimeDiscreteDGscheme,
)
from scimba_jax.linear_approximation.variables.variables_dg import VariablesDG
from scimba_jax.nonlinear_approximation.model_class.funcparam_vectorial import (
    ParamVecFunction,
)
from scimba_jax.physical_models.classical_weakform.diffusion_advection_reaction_weak_form import (
    EllipticWeakForm,
)

### The problem data


In [ ]:
DISK_CENTER = (0.0, 0.0)
DISK_RADIUS = 1.0
DOM_X = Disk2D(DISK_CENTER, DISK_RADIUS, is_main_domain=True)
DOM_T = (0.0, 0.5)
NT = 40
DT = (DOM_T[1] - DOM_T[0]) / NT


def advection_field(x: jnp.ndarray) -> jnp.ndarray:
    """Rigid rotation velocity field a(x) = (-2*pi*x2, 2*pi*x1).

    Uses scalar components so it works both on batches of points, x of shape
    (n, 2), and on single points, x of shape (2,), as the residuals are
    evaluated pointwise under a vmap.
    """
    x1, x2 = x[..., 0], x[..., 1]
    return jnp.stack([-2.0 * jnp.pi * x2, 2.0 * jnp.pi * x1], axis=-1)


def exact_sol(t: jnp.ndarray, x: jnp.ndarray) -> jnp.ndarray:
    """Exact solution: the initial bump rotated by the angle 2*pi*t."""
    t = jnp.atleast_1d(t)
    angle = -2.0 * jnp.pi * t[..., 0]
    x1, x2 = x[..., 0], x[..., 1]
    x1_0 = jnp.cos(angle) * x1 - jnp.sin(angle) * x2
    x2_0 = jnp.sin(angle) * x1 + jnp.cos(angle) * x2
    return jnp.exp(-150.0 * ((x1_0 - 0.3) ** 2 + x2_0**2))[..., None]


def f_init(x: jnp.ndarray) -> jnp.ndarray:
    """Initial condition."""
    t = jnp.zeros_like(x[..., 0:1])
    return exact_sol(t, x)


def exact_characteristic_foot(t: jnp.ndarray, x: jnp.ndarray, dt: float) -> jnp.ndarray:
    """Foot of the characteristic: rotate x backwards by the angle 2*pi*dt."""
    angle = -2.0 * jnp.pi * dt
    x1, x2 = x[0], x[1]
    x1_foot = jnp.cos(angle) * x1 - jnp.sin(angle) * x2
    x2_foot = jnp.sin(angle) * x1 + jnp.cos(angle) * x2
    return jnp.stack([x1_foot, x2_foot])


def u_exact_final(x: jnp.ndarray) -> jnp.ndarray:
    """Exact solution at the final time, as a function of x only."""
    return exact_sol(jnp.array(DOM_T[1]), x).reshape(OUT_DIM)

### The neural methods and the DG method

We build the three neural methods, a PINN, a Discrete PINN and a Neural
Semi-Lagrangian method, plus a classical DG method time-stepped with
Runge-Kutta. The neural methods are trained on the same domain and with the
same number of collocation points, so that the comparison is fair.


In [ ]:
N_COLLOC = 2000
N_BC_COLLOC = 1000
N_IC_COLLOC = 1000
N_EPOCHS_INIT = 200
N_EPOCHS = 20
TRANSPORT_HIDDEN = [16, 16]

sampler_t = TensorizedSampler([DomainSampler(DOM_X)], model_type="x")
# the PINN takes time as an input, so its sampler needs a time variable; this
# is also what makes it draw initial-condition points ("ic interior")
sampler_pinn = TensorizedSampler(
    [UniformTimeSampler(DOM_T), DomainSampler(DOM_X)],
    model_type="t_x",
    bc=True,
    ic=True,
)


# --- PINN ---
def make_pinn_transport(hidden_sizes, n_epochs, seed=0):
    """Build and train a PINN for the rotating advection equation."""
    key = jax.random.PRNGKey(seed)
    nn = MLP(in_size=3, out_size=1, hidden_sizes=hidden_sizes, key=key)
    space = ApproximationSpace({"x": 2}, [(nn, "scalar", None)], model_type="t_x")
    model = AdvectionND(
        main_domain=DOM_X,
        time_domain=DOM_T,
        velocity=advection_field,
        # no boundary condition needed: the exact solution vanishes on the
        # boundary (the bump stays inside the disk), so "strong" with the
        # default identity post-processing simply drops the boundary term
        bc="strong",
        ic="weak",
        f_ic_rhs=f_init,
    )
    pinn = Projector(model, space, sampler_pinn)
    start = timeit.default_timer()
    key, pinn = pinn.project(key, space, n_epochs, N_COLLOC, N_BC_COLLOC, N_IC_COLLOC)
    elapsed = timeit.default_timer() - start
    return pinn, elapsed


# --- Discrete PINN ---
def make_discrete_pinn(hidden_sizes, n_epochs, seed=0):
    """Build and train a Discrete PINN for the rotating advection equation."""
    key = jax.random.PRNGKey(seed)
    nn = MLP(in_size=2, out_size=1, hidden_sizes=hidden_sizes, key=key)
    space = ApproximationSpace({"x": 2}, [(nn, "scalar", None)], model_type="x")

    # strong BC (no boundary term, see above); fully implicit time stepping
    # (Alexander's L-stable SDIRK), so the scheme is stable for any dt = T/NT
    pde = SteadyAdvectionND(DOM_X, DOM_T, advection_field, bc="strong")
    dpinn = DiscretePINN(
        DOM_X, DOM_T, sampler_t, out_size=1, nt=NT,
        butcher_tableau=build_alexander_tableau(),
        explicit_pde=None,
        implicit_pde=pde,
        exact_solution=exact_sol,
    )
    start = timeit.default_timer()
    key, dpinn = dpinn.initialize(key, space, f_init, N_EPOCHS_INIT, N_COLLOC,
                                  n_bc_colloc=N_BC_COLLOC)
    space = dpinn.space
    key, dpinn = dpinn.solve(key, space, n_epochs, N_COLLOC, n_bc_colloc=N_BC_COLLOC)
    elapsed = timeit.default_timer() - start
    return dpinn, elapsed


# --- Neural Semi-Lagrangian ---
def make_neural_sl(hidden_sizes, n_epochs, seed=0):
    """Build and train a Neural Semi-Lagrangian method for the rotating advection."""
    key = jax.random.PRNGKey(seed)
    nn = MLP(in_size=2, out_size=1, hidden_sizes=hidden_sizes, key=key)
    space = ApproximationSpace({"x": 2}, [(nn, "scalar", None)], model_type="x")

    nsl = NeuralSemiLagrangian(
        main_domain=DOM_X,
        time_domain=DOM_T,
        sampler=sampler_t,
        dt=DT,
        advection_field=advection_field,
        exact_characteristic_foot=exact_characteristic_foot,
        periodic=False,
        out_size=1,
        model_type="x",
        exact_solution=exact_sol,
    )
    start = timeit.default_timer()
    key, nsl = nsl.initialize(key, space, f_init, N_EPOCHS_INIT, N_COLLOC)
    space = nsl.space
    key, nsl = nsl.solve(key, space, n_epochs, N_COLLOC)
    elapsed = timeit.default_timer() - start
    return nsl, elapsed


# --- DG ---
def taylor(coords, i, mesh):
    return local_taylor_basis(coords, i, mesh, order=POLY_ORDER, out_dim=OUT_DIM)


def taylor_by_logical(y, i, mesh):
    return local_taylor_basis_by_logical(y, i, mesh, order=POLY_ORDER, out_dim=OUT_DIM)


def make_dg_transport(n_cells, poly_order):
    """Build and solve a time-discrete DG scheme for the rotating advection.

    The DG space is a Taylor basis (discontinuous across cell interfaces) on
    the same disk meshes as in the first part, the advection is discretized
    with an upwind flux, and time is integrated with Alexander's implicit
    L-stable scheme, so any dt = T/NT is stable. The stage linear systems are
    non-symmetric, hence GMRES.
    """
    global POLY_ORDER
    POLY_ORDER = poly_order
    mesh = load_disk_mesh(n_cells)
    basis = AnalyticBasis(
        nb_basis=(poly_order + 1) ** PHYSICAL_DIM,
        out_dim=OUT_DIM,
        mesh=mesh,
        local_basis=taylor,
        local_basis_by_logical=taylor_by_logical,
        basis_type="scalar",
    )
    variables = VariablesDG(basis=basis, nb_variables=OUT_DIM)
    pde = EllipticWeakForm(dim=PHYSICAL_DIM, A=None, b=advection_field, c=None, f=None)
    scheme = TimeDiscreteDGscheme(
        pde,
        variables,
        UpwindFlux(),
        butcher_tableau=build_alexander_tableau(),
        dt=DT,
        dirichlet=zero_bc,
        cg_solver="gmres",
        max_iter_linear=200,
    )
    dofsl = scheme.initialize(f_init)
    # Drive the time loop ourselves, with the scheme closed over (hence static)
    # in the jitted step: passing the scheme as a jitted argument mis-assembles
    # the stage operators.
    step = jax.jit(lambda d, t: scheme.step(d, t))
    ts = DOM_T[0] + DT * jnp.arange(NT)
    dofsl, _ = jax.lax.scan(lambda d, t: (step(d, t), None), dofsl, ts)
    scheme.variables.dofsl = dofsl
    return scheme

### Comparing DoF, error and time

We train each neural method with a fixed network size, and sweep the DG
method over the mesh resolutions of the first part; for each configuration
we record the number of degrees of freedom, the final relative $L^2$ error
and the computation time.


In [ ]:
transport_results = []

# --- PINN ---
pinn, elapsed = make_pinn_transport(TRANSPORT_HIDDEN, N_EPOCHS)
ndof = pinn.space.ndof
key = jax.random.PRNGKey(42)
n_pts = 20000
xy = jax.random.uniform(key, (n_pts, 2), minval=-1.0, maxval=1.0)
# keep only points inside the disk
mask = xy[:, 0] ** 2 + xy[:, 1] ** 2 <= 1.0
xy = xy[mask]
t_final = jnp.ones((xy.shape[0], 1)) * DOM_T[1]
u_num = pinn.evaluate(t_final, xy)
u_ex = exact_sol(t_final, xy)
err = float(jnp.linalg.norm(u_num - u_ex) / jnp.linalg.norm(u_ex))
transport_results.append(("PINN", ndof, err, elapsed))
print(f"PINN: ndof={ndof:>6}  err={err:.3e}  time={elapsed:.3f}s\n")

# --- Discrete PINN ---
dpinn, elapsed = make_discrete_pinn(TRANSPORT_HIDDEN, N_EPOCHS)
ndof = dpinn.space.ndof
errors = dpinn.errors_over_time
err = float(errors[-1, 0])
transport_results.append(("Discrete PINN", ndof, err, elapsed))
print(f"Discrete PINN: ndof={ndof:>6}  err={err:.3e}  time={elapsed:.3f}s\n")

# --- Neural Semi-Lagrangian ---
nsl, elapsed = make_neural_sl(TRANSPORT_HIDDEN, N_EPOCHS)
ndof = nsl.space.ndof
errors = nsl.errors_over_time
err = float(errors[-1, 0])
transport_results.append(("Neural SL", ndof, err, elapsed))
print(f"Neural SL: ndof={ndof:>6}  err={err:.3e}  time={elapsed:.3f}s\n")

# --- DG (swept over the same meshes as in the first part) ---
for n_cells in N_CELLS_SWEEP:
    t0 = timeit.default_timer()
    dg = make_dg_transport(n_cells, 1)
    t_dg = timeit.default_timer() - t0
    ndof = dg.variables.dofsl.size
    err = float(l2_error(dg, u_exact_final, relative=True))
    transport_results.append((f"DG {n_cells}x{n_cells}", ndof, err, t_dg))
    print(f"DG   {n_cells}x{n_cells}: ndof={ndof:>6}  err={err:.3e}  time={t_dg:.3f}s\n")

### Final solutions and pointwise errors

We evaluate each method at the final time on a cloud of points in the disk,
and plot the numerical solution, the exact solution and the absolute error.


In [ ]:
def eval_nn(space, *args):
    """Evaluate a neural approximation space on batched points."""
    variables = space.create_variables()
    batched = ParamVecFunction.cat(variables).vmap_on_physical_variables()
    return batched(space, *args)


key = jax.random.PRNGKey(0)
n_pts = 20000
r = jnp.sqrt(jax.random.uniform(key, (n_pts, 1)))
theta = jax.random.uniform(key, (n_pts, 1), minval=0.0, maxval=2.0 * jnp.pi)
xy = jnp.concatenate([r * jnp.cos(theta), r * jnp.sin(theta)], axis=1)
t_final = jnp.ones((n_pts, 1)) * DOM_T[1]
u_ex = exact_sol(t_final, xy)

solutions = {
    "PINN": pinn.evaluate(t_final, xy),
    "Discrete PINN": eval_nn(dpinn.space, xy),
    "Neural SL": eval_nn(nsl.space, xy),
    f"DG {N_CELLS_SWEEP[-1]}x{N_CELLS_SWEEP[-1]}": dg.variables.evaluate(xy),
}

fig, axes = plt.subplots(len(solutions), 3, figsize=(13, 3.2 * len(solutions)))
for i, (name, u_num) in enumerate(solutions.items()):
    for j, (vals, title) in enumerate(
        [(u_num, "numerical"), (u_ex, "exact"), (jnp.abs(u_num - u_ex), "abs. error")]
    ):
        ax = axes[i, j]
        cf = ax.tricontourf(xy[:, 0], xy[:, 1], vals[:, 0], levels=20, cmap="viridis")
        fig.colorbar(cf, ax=ax)
        ax.set_title(f"{name} — {title}" if j == 0 else title)
        ax.set_aspect("equal")
plt.tight_layout()
plt.show()

### Plotting the results

We plot the final error and the computation time of each method as a bar chart.


In [ ]:
methods = [r[0] for r in transport_results]
errors = [r[2] for r in transport_results]
times = [r[3] for r in transport_results]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.bar(methods, errors, color="steelblue")
ax.set_ylabel("Relative L2 error (final time)")
ax.set_title("Final error")
ax.tick_params(axis="x", rotation=15)

ax = axes[1]
ax.bar(methods, times, color="coral")
ax.set_ylabel("Computation time (s)")
ax.set_title("Computation time")
ax.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()